# 碩論深度學習主分析：

本 Notebook 執行 LSTM-NARX、TCN 與 Euler-based ODE-LSTM，分別預測 HbA1c、Glu-AC、LDL 與 TG。

正式規則：

- 輸入為病患層級切分後建立的單一 `prediction-safe dataset`（`.imp = 1`）。
- outcome 不做跨資料集插補；歷史序列與預測目標僅使用原始觀測值。
- BMI 缺失由上游 R 程式以 train set 估計之分組／整體中位數處理。
- 每位可評估病患以各 split 最後一次原始觀測結果作為 target；模型只使用 target 之前資料。
- 實證預測表報告單一 RMSE、MAE、MAPE 與 Bias，不再報告插補間平均值 ± SD。
- 五模型之病患層級 bootstrap 95% 信賴區間由後續 `資料分析_方案A_最終完整版.R` 依個體預測結果統一計算。
- LSTM-NARX 與 TCN 使用 outcome 與 padding indicator 雙通道；ODE-LSTM 使用顯式 structural mask。
- ODE-LSTM 採 Euler 近似進行連續時間隱藏狀態轉移，不宣稱完整估計個體層級生理 ODE。


In [1]:
# ============================================================
# 0. 套件、版本與固定設定
# ============================================================

import os
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
os.environ.setdefault("PYTHONHASHSEED", "5014")

import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate, Dropout
from tensorflow.keras.callbacks import EarlyStopping

try:
    from tcn import TCN
except Exception as exc:
    raise ImportError(
        "本論文設定使用 keras-tcn 的 TCN layer，不再自動改用不同架構。"
        "請先執行：pip install keras-tcn"
    ) from exc

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SCRIPT_VERSION = "2026-07-15-dl-scheme-A-final-v9-output-cleanup"
SEED = 5014
EXPECTED_M = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(SEED)

# ------------------------------------------------------------
# 路徑
# ------------------------------------------------------------

CANDIDATE_DATA_PATHS = [
    Path("analysis_outputs") / "dl_data" / "prediction_long_with_split_type2.csv",
    Path("dl_data") / "prediction_long_with_split_type2.csv",
    Path("prediction_long_with_split_type2.csv"),
]

DATA_PATH = next((p for p in CANDIDATE_DATA_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "找不到 prediction_long_with_split_type2.csv。"
        "請先完成最新版表格.R。"
    )

OUT_DIR = Path("analysis_outputs") / "dl_output"
TABLE_DIR = OUT_DIR / "tables"
FIGURE_DIR = OUT_DIR / "figures"
MODEL_DIR = OUT_DIR / "saved_models"

for d in [OUT_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 為避免資料分析.R 誤讀舊結果，正式重跑前先移除主要舊輸出。
STALE_FILES = [
    OUT_DIR / "dl_prediction_metrics_summary_group.csv",
    TABLE_DIR / "dl_prediction_metrics_summary_group.csv",
    TABLE_DIR / "dl_prediction_metrics_by_imp_group.csv",
    TABLE_DIR / "dl_test_predictions_individual.csv",
    TABLE_DIR / "dl_loss_history_by_epoch.csv",
    TABLE_DIR / "dl_sequence_sample_size.csv",
    TABLE_DIR / "dl_scaler_parameters.csv",
    TABLE_DIR / "dl_target_manifest.csv",
    TABLE_DIR / "dl_target_truth_consistency_qc.csv",
    TABLE_DIR / "dl_ode_target_gap_summary.csv",
    TABLE_DIR / "dl_mape_denominator_qc.csv",
    TABLE_DIR / "dl_model_settings.csv",
    TABLE_DIR / "dl_trajectory_interval_counts_and_means.csv",
    TABLE_DIR / "DL_model_outputs_type2.xlsx",
]

for p in STALE_FILES:
    if p.exists():
        p.unlink()

# 清除舊版逐模型／逐 outcome 的重複輸出。
# 本版不再產生這些檔案。
STALE_PATTERNS = [
    "eval_*.csv",
    "metrics_by_imp_group_*.csv",
]

for pattern in STALE_PATTERNS:
    for p in TABLE_DIR.glob(pattern):
        if p.is_file():
            p.unlink()

# ------------------------------------------------------------
# 模型設定
# ------------------------------------------------------------

OUTCOMES = ["HbA1c", "Glu_AC", "LDL", "TG"]

OUTCOME_LABELS = {
    "HbA1c": "HbA1c",
    "Glu_AC": "Glu-AC",
    "LDL": "LDL",
    "TG": "TG",
}

OBSERVED_FLAG_MAP = {
    "HbA1c": "HbA1c_observed",
    "Glu_AC": "Glu_AC_observed",
    "LDL": "LDL_observed",
    "TG": "TG_observed",
}

MODEL_ORDER = ["LSTM-NARX", "TCN", "ODE-LSTM"]

GROUP_LEVELS = ["Overall", "Control", "Statin"]
GROUP_LABELS_ZH = {
    "Overall": "整體",
    "Control": "控制組",
    "Statin": "史他汀組",
}
GROUP_LABELS_EN = {
    "Overall": "Overall",
    "Control": "Control group",
    "Statin": "Statin group",
}

AGE_VAR = "Age_baseline"
BMI_VAR = "BMI"

MAX_LEN = 7
EPOCHS = 100
BATCH_SIZE = 32
PATIENCE = 10
LEARNING_RATE_TORCH = 1e-3
MAX_ODE_DT = 5.0
MIN_PLOT_TARGET_N = 5

plt.rcParams["font.sans-serif"] = [
    "Microsoft JhengHei", "Noto Sans CJK TC", "Noto Sans CJK JP",
    "SimHei", "Arial Unicode MS", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

print("程式版本：", SCRIPT_VERSION)
print("資料來源：", DATA_PATH)
print("輸出資料夾：", OUT_DIR)
print("運算裝置：", DEVICE)


程式版本： 2026-07-15-dl-scheme-A-final-v9-output-cleanup
資料來源： analysis_outputs\dl_data\prediction_long_with_split_type2.csv
輸出資料夾： analysis_outputs\dl_output
運算裝置： cpu


In [2]:
# ============================================================
# 1. 共用工具函數
# ============================================================

def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename_map = {}
    if "HbA1C" in df.columns and "HbA1c" not in df.columns:
        rename_map["HbA1C"] = "HbA1c"
    if "Glucose_AC" in df.columns and "Glu_AC" not in df.columns:
        rename_map["Glucose_AC"] = "Glu_AC"
    if "Triglycerides" in df.columns and "TG" not in df.columns:
        rename_map["Triglycerides"] = "TG"
    return df.rename(columns=rename_map)


def parse_bool_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.astype(bool)
    x = s.astype(str).str.strip().str.lower()
    out = x.map({
        "true": True, "t": True, "1": True, "yes": True,
        "false": False, "f": False, "0": False, "no": False,
    })
    if out.isna().any():
        bad = sorted(s[out.isna()].astype(str).unique().tolist())
        raise ValueError(f"觀測旗標含無法辨識值：{bad[:10]}")
    return out.astype(bool)


def safe_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def make_log_col(outcome: str) -> str:
    return f"log_{outcome}"


def make_z_col(col: str) -> str:
    return f"{col}_z"


def regression_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    keep = np.isfinite(y_true) & np.isfinite(y_pred) & (y_true > 0)
    yt = y_true[keep]
    yp = y_pred[keep]

    if len(yt) == 0:
        return {
            "RMSE": np.nan,
            "MAE": np.nan,
            "MAPE": np.nan,
            "Bias": np.nan,
            "n_test": 0,
        }

    err = yp - yt
    return {
        "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
        "MAE": float(mean_absolute_error(yt, yp)),
        "MAPE": float(np.mean(np.abs(err / yt)) * 100),
        "Bias": float(np.mean(err)),
        "n_test": int(len(yt)),
    }


def metrics_by_group(pred_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for group_eval in GROUP_LEVELS:
        sub = pred_df if group_eval == "Overall" else pred_df[pred_df["Group"] == group_eval]
        m = regression_metrics(sub["y_true"], sub["y_pred"])
        rows.append({
            ".imp": int(pred_df[".imp"].iloc[0]),
            "outcome": str(pred_df["outcome"].iloc[0]),
            "outcome_label": str(pred_df["outcome_label"].iloc[0]),
            "model": str(pred_df["model"].iloc[0]),
            "Group_eval": group_eval,
            "Group_label": GROUP_LABELS_EN[group_eval],
            **m,
        })
    return pd.DataFrame(rows)


def summarize_single_dataset_metrics(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """整理單一 prediction-safe dataset 的點估計，不計算插補間 SD。"""
    if metrics_df.empty:
        return pd.DataFrame()

    required = [
        "outcome", "outcome_label", "model", "Group_eval", "Group_label",
        "RMSE", "MAE", "MAPE", "Bias", "n_test",
    ]
    missing = [x for x in required if x not in metrics_df.columns]
    if missing:
        raise ValueError(f"預測指標缺少欄位：{missing}")

    if metrics_df.duplicated(
        ["outcome", "model", "Group_eval"]
    ).any():
        raise RuntimeError("單一 prediction-safe dataset 出現重複模型／指標／分組結果。")

    out = metrics_df[required].copy()
    out["dataset_n"] = 1
    out["outcome"] = pd.Categorical(out["outcome"], OUTCOMES, ordered=True)
    out["model"] = pd.Categorical(out["model"], MODEL_ORDER, ordered=True)
    out["Group_eval"] = pd.Categorical(
        out["Group_eval"], GROUP_LEVELS, ordered=True
    )
    return out.sort_values(
        ["outcome", "model", "Group_eval"]
    ).reset_index(drop=True)


def pad_left_1d(seq, max_len: int, pad_value: float = 0.0) -> np.ndarray:
    arr = np.asarray(seq, dtype=float).reshape(-1)
    if len(arr) >= max_len:
        return arr[-max_len:]
    pad = np.full(max_len - len(arr), pad_value, dtype=float)
    return np.concatenate([pad, arr])


def model_run_seed(model_name: str, outcome: str) -> int:
    # 每一 model/outcome 使用固定初始化種子，以提高分析重現性。
    return (
        SEED
        + 100 * OUTCOMES.index(outcome)
        + 10 * MODEL_ORDER.index(model_name)
    )


In [3]:
# ============================================================
# 2. 讀取、欄位標準化與輸入資料 QC
# ============================================================

def preprocess_raw_data(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_column_names(df).copy()

    required_cols = [
        ".imp", "row_id", "ID", "split", "Group",
        "Time_in_Years", "Time_Category",
        "Sex", AGE_VAR, BMI_VAR,
    ] + OUTCOMES + list(OBSERVED_FLAG_MAP.values())

    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError("資料缺少必要欄位：" + ", ".join(missing_cols))

    df[".imp"] = pd.to_numeric(df[".imp"], errors="raise").astype(int)
    df["row_id"] = pd.to_numeric(df["row_id"], errors="raise").astype(int)
    df["ID"] = df["ID"].astype(str)
    df["split"] = df["split"].astype(str)
    df["Group"] = df["Group"].astype(str)

    valid_splits = {"train", "validation", "test"}
    if set(df["split"].dropna().unique()) != valid_splits:
        raise ValueError(f"split 類別不符：{sorted(df['split'].dropna().unique())}")

    valid_groups = {"Control", "Statin"}
    if set(df["Group"].dropna().unique()) != valid_groups:
        raise ValueError(f"Group 類別不符：{sorted(df['Group'].dropna().unique())}")

    df["Group_Statin"] = (df["Group"] == "Statin").astype(float)

    sex_chr = df["Sex"].astype(str).str.strip()
    df["Sex_num"] = np.select(
        [
            sex_chr.isin(["0", "Female", "female", "F", "f", "女"]),
            sex_chr.isin(["1", "Male", "male", "M", "m", "男"]),
        ],
        [0.0, 1.0],
        default=np.nan,
    )
    if df["Sex_num"].isna().any():
        bad = sorted(sex_chr[df["Sex_num"].isna()].unique().tolist())
        raise ValueError(f"Sex 含未知編碼：{bad[:10]}")

    for flag in OBSERVED_FLAG_MAP.values():
        df[flag] = parse_bool_series(df[flag])

    for col in ["Time_in_Years", AGE_VAR, BMI_VAR] + OUTCOMES:
        df[col] = safe_numeric(df[col])

    # prediction-safe 資料允許 outcome 缺失；缺失 outcome 不進入 target 或歷史序列。
    for col in [AGE_VAR, BMI_VAR]:
        if df[col].isna().any():
            raise ValueError(f"prediction-safe 資料仍有缺失：{col}")
    if df["Time_in_Years"].isna().any():
        raise ValueError("Time_in_Years 仍有缺失。")

    for outcome in OUTCOMES:
        observed_values = df[outcome].dropna()
        if (observed_values <= 0).any():
            raise ValueError(f"{outcome} 的實際觀測值含非正值，無法取自然對數。")
        df[make_log_col(outcome)] = np.where(
            df[outcome].notna(),
            np.log(df[outcome]),
            np.nan,
        )

    if (df[BMI_VAR] <= 0).any():
        raise ValueError("BMI 含非正值。")

    df["Time_Category"] = df["Time_Category"].astype(str)
    df["Time_Num"] = pd.to_numeric(
        df["Time_Category"].str.replace("T", "", regex=False),
        errors="coerce",
    )
    if df["Time_Num"].isna().any() or not df["Time_Num"].between(1, 19).all():
        raise ValueError("Time_Category 並非完整限制於 T1–T19。")

    df = df.sort_values(
        [".imp", "ID", "Time_in_Years", "row_id"]
    ).reset_index(drop=True)

    df["Delta_Time"] = (
        df.groupby([".imp", "ID"], sort=False)["Time_in_Years"]
        .diff()
        .fillna(0.0)
        .clip(lower=0.0)
    )

    return df


raw_df = pd.read_csv(DATA_PATH)
dt_all = preprocess_raw_data(raw_df)

# 記錄上游資料用途說明；若欄位存在，禁止誤寫為 training-only MICE。
data_role_values = (
    sorted(raw_df["data_role"].dropna().astype(str).unique().tolist())
    if "data_role" in raw_df.columns
    else []
)

# ------------------------------------------------------------
# 單一 prediction-safe dataset 之結構與一致性檢查
# ------------------------------------------------------------
# 注意：
# `.imp = 1` 僅為維持既有資料結構相容性，
# 不代表第一組 MICE 完成資料。
# 本 DL 分析僅使用一個 deterministic prediction-safe dataset。

qc_rows = []

prediction_dataset_n = int(dt_all[".imp"].nunique())

qc_rows.append({
    "check": "number_of_prediction_datasets",
    "value": prediction_dataset_n,
    "expected": EXPECTED_M,
    "passed": prediction_dataset_n == EXPECTED_M,
})

duplicate_row_id_n = int(
    dt_all.duplicated([".imp", "row_id"]).sum()
)

qc_rows.append({
    "check": "duplicate_row_id_within_prediction_dataset",
    "value": duplicate_row_id_n,
    "expected": 0,
    "passed": duplicate_row_id_n == 0,
})

row_count_by_dataset = (
    dt_all
    .groupby(".imp", sort=True)
    .size()
)

qc_rows.append({
    "check": "row_count_structure_valid",
    "value": int(row_count_by_dataset.nunique()),
    "expected": 1,
    "passed": int(row_count_by_dataset.nunique()) == 1,
})

row_id_dataset_n = (
    dt_all
    .groupby("row_id", sort=False)[".imp"]
    .nunique()
)

row_id_not_in_all_datasets = int(
    (row_id_dataset_n != EXPECTED_M).sum()
)

qc_rows.append({
    "check": "row_id_present_in_all_prediction_datasets",
    "value": row_id_not_in_all_datasets,
    "expected": 0,
    "passed": row_id_not_in_all_datasets == 0,
})

patient_split_n = (
    dt_all[
        [".imp", "ID", "split"]
    ]
    .drop_duplicates()
    .groupby(
        [".imp", "ID"]
    )["split"]
    .nunique()
)

patient_multiple_split_n = int(
    (patient_split_n != 1).sum()
)

qc_rows.append({
    "check": "patient_multiple_split_within_prediction_dataset",
    "value": patient_multiple_split_n,
    "expected": 0,
    "passed": patient_multiple_split_n == 0,
})

split_across_datasets = (
    dt_all[
        ["ID", "split"]
    ]
    .drop_duplicates()
    .groupby("ID")["split"]
    .nunique()
)

split_inconsistent_n = int(
    (split_across_datasets != 1).sum()
)

qc_rows.append({
    "check": "patient_split_inconsistent_across_prediction_datasets",
    "value": split_inconsistent_n,
    "expected": 0,
    "passed": split_inconsistent_n == 0,
})

group_across_datasets = (
    dt_all[
        ["ID", "Group"]
    ]
    .drop_duplicates()
    .groupby("ID")["Group"]
    .nunique()
)

group_inconsistent_n = int(
    (group_across_datasets != 1).sum()
)

qc_rows.append({
    "check": "patient_group_inconsistent_across_prediction_datasets",
    "value": group_inconsistent_n,
    "expected": 0,
    "passed": group_inconsistent_n == 0,
})

for flag in OBSERVED_FLAG_MAP.values():
    n_missing = int(
        dt_all[flag].isna().sum()
    )

    qc_rows.append({
        "check": f"{flag}_parse_missing",
        "value": n_missing,
        "expected": 0,
        "passed": n_missing == 0,
    })

    flag_consistency = (
        dt_all[
            ["row_id", flag]
        ]
        .drop_duplicates()
        .groupby("row_id")[flag]
        .nunique(dropna=False)
    )

    n_bad = int(
        (flag_consistency != 1).sum()
    )

    qc_rows.append({
        "check": (
            f"{flag}_inconsistent_"
            "across_prediction_datasets"
        ),
        "value": n_bad,
        "expected": 0,
        "passed": n_bad == 0,
    })

# 原始觀測值必須在 prediction-safe dataset 中保持一致。
# 此檢查直接使用原始尺度，不經 log、標準化或反轉換。
observed_truth_qc_rows = []

for outcome, flag in OBSERVED_FLAG_MAP.items():
    truth_one = (
        dt_all.loc[
            dt_all[flag],
            [
                ".imp",
                "row_id",
                "ID",
                "Group",
                outcome,
            ],
        ]
        .groupby(
            [
                "row_id",
                "ID",
                "Group",
            ],
            as_index=False,
        )
        .agg(
            y_true_min=(
                outcome,
                "min",
            ),
            y_true_max=(
                outcome,
                "max",
            ),
            n_dataset=(
                ".imp",
                "nunique",
            ),
        )
    )

    truth_one["outcome"] = outcome

    truth_one["truth_range"] = (
        truth_one["y_true_max"]
        - truth_one["y_true_min"]
    )

    truth_one["passed"] = (
        truth_one["truth_range"]
        .abs()
        .le(1e-12)
        & truth_one["n_dataset"]
        .eq(EXPECTED_M)
    )

    observed_truth_qc_rows.append(
        truth_one
    )

    n_bad = int(
        (~truth_one["passed"]).sum()
    )

    qc_rows.append({
        "check": (
            f"{outcome}_observed_value_"
            "inconsistent_across_prediction_datasets"
        ),
        "value": n_bad,
        "expected": 0,
        "passed": n_bad == 0,
    })

observed_truth_qc = pd.concat(
    observed_truth_qc_rows,
    ignore_index=True,
)

observed_truth_qc.to_csv(
    TABLE_DIR
    / "dl_input_observed_truth_consistency_qc.csv",
    index=False,
    encoding="utf-8-sig",
)

input_qc = pd.DataFrame(
    qc_rows
)

input_qc.to_csv(
    TABLE_DIR
    / "dl_input_qc.csv",
    index=False,
    encoding="utf-8-sig",
)

if not input_qc["passed"].all():
    raise RuntimeError(
        "DL 輸入資料 QC 未通過：\n"
        + input_qc.loc[
            ~input_qc["passed"]
        ].to_string(index=False)
    )

same_time_qc = (
    dt_all.loc[
        dt_all[".imp"] == 1
    ]
    .groupby(
        [
            "ID",
            "Time_in_Years",
        ],
        as_index=False,
    )
    .agg(
        n_rows=(
            "row_id",
            "size",
        )
    )
    .query(
        "n_rows > 1"
    )
)

same_time_qc.to_csv(
    TABLE_DIR
    / "dl_same_patient_same_time_qc.csv",
    index=False,
    encoding="utf-8-sig",
)

print("讀取完成。")
print("prediction-safe 資料集數：", prediction_dataset_n,)
print("總列數：", len(dt_all),)
print("prediction dataset 1 病患數：", dt_all.loc[dt_all[".imp"] == 1, "ID",].nunique(),)
print(input_qc)


讀取完成。
prediction-safe 資料集數： 1
總列數： 17252
prediction dataset 1 病患數： 1790
                                                check  value  expected  passed
0                       number_of_prediction_datasets      1         1    True
1          duplicate_row_id_within_prediction_dataset      0         0    True
2                           row_count_structure_valid      1         1    True
3           row_id_present_in_all_prediction_datasets      0         0    True
4    patient_multiple_split_within_prediction_dataset      0         0    True
5   patient_split_inconsistent_across_prediction_d...      0         0    True
6   patient_group_inconsistent_across_prediction_d...      0         0    True
7                        HbA1c_observed_parse_missing      0         0    True
8   HbA1c_observed_inconsistent_across_prediction_...      0         0    True
9                       Glu_AC_observed_parse_missing      0         0    True
10  Glu_AC_observed_inconsistent_across_prediction...      

In [4]:
# ============================================================
# 3. Train-only 標準化與病患序列建立
# ============================================================

def fit_scaler_from_train(df_train: pd.DataFrame, outcome: str) -> dict:
    # 只使用 train set 估計對數 outcome 與連續共變數之標準化參數。
    cols = [make_log_col(outcome), "Time_in_Years", AGE_VAR, BMI_VAR]
    scaler = {}
    for col in cols:
        x = pd.to_numeric(df_train[col], errors="coerce")
        mu = float(x.mean())
        sd = float(x.std(ddof=1))
        if not np.isfinite(mu):
            raise ValueError(f"train set 無法估計 {col} 平均數。")
        if not np.isfinite(sd) or sd <= 0:
            sd = 1.0
        scaler[col] = {"mean": mu, "sd": sd}
    return scaler


def apply_scaler(df: pd.DataFrame, scaler: dict, outcome: str) -> pd.DataFrame:
    out = df.copy()
    for col in [make_log_col(outcome), "Time_in_Years", AGE_VAR, BMI_VAR]:
        out[make_z_col(col)] = (
            out[col] - scaler[col]["mean"]
        ) / scaler[col]["sd"]
    return out


def inverse_transform_y(y_std, scaler: dict, outcome: str) -> np.ndarray:
    y_std = np.asarray(y_std, dtype=float)
    info = scaler[make_log_col(outcome)]
    return np.exp(y_std * info["sd"] + info["mean"])


def model_exog_cols(model_name: str) -> list[str]:
    common = [
        make_z_col(AGE_VAR),
        "Sex_num",
        make_z_col(BMI_VAR),
        "Group_Statin",
    ]
    if model_name in {"LSTM-NARX", "TCN"}:
        return [make_z_col("Time_in_Years")] + common
    if model_name == "ODE-LSTM":
        # ODE-LSTM 已用 Δt 描述不等間隔，不重複納入目標追蹤時間。
        return common
    raise ValueError(f"未知模型：{model_name}")


def build_sequence_dataset(
    df_scaled: pd.DataFrame,
    split_name: str,
    outcome: str,
    model_name: str,
    max_len: int = MAX_LEN,
    min_history: int = 1,
):
    """
    每位病患於該 split 建立一個最後一次「原始觀測」target。

    - target 必須 observed_flag == TRUE。
    - target 前歷史 outcome 僅使用原始實際觀測值；prediction-safe 資料不插補 outcome。
    - LSTM-NARX／TCN：X_seq 含標準化對數 outcome 與明確 padding mask 通道。
    - ODE-LSTM：另回傳重算後的歷史 Δt、最後歷史觀測至 target 的 Δt，
      以及 padding mask。
    """
    y_z_col = make_z_col(make_log_col(outcome))
    observed_col = OBSERVED_FLAG_MAP[outcome]
    exog_cols = model_exog_cols(model_name)

    df_split = (
        df_scaled[df_scaled["split"] == split_name]
        .copy()
        .sort_values(["ID", "Time_in_Years", "row_id"])
    )

    X_seq, X_dt, X_target_dt, X_exog, y, mask, meta = [], [], [], [], [], [], []

    for pid, g in df_split.groupby("ID", sort=False):
        g = g.sort_values(["Time_in_Years", "row_id"]).reset_index(drop=True)

        if len(g) <= min_history:
            continue

        needed_target = [y_z_col, "Time_in_Years"] + exog_cols
        valid_target = (
            g[observed_col].astype(bool)
            & g[needed_target].notna().all(axis=1)
        ).to_numpy()

        candidate_pos = np.where(valid_target)[0]
        candidate_pos = candidate_pos[candidate_pos >= min_history]
        if len(candidate_pos) == 0:
            continue

        target_pos = int(candidate_pos[-1])
        target = g.iloc[target_pos].copy()
        hist = g.iloc[:target_pos].copy()

        hist = hist.dropna(subset=[y_z_col, "Time_in_Years"])
        if len(hist) < min_history:
            continue

        # 只保留最接近 target 的最後 max_len 筆歷史。
        hist = hist.tail(max_len).copy()

        hist_y = hist[y_z_col].to_numpy(dtype=float)
        hist_time = hist["Time_in_Years"].to_numpy(dtype=float)

        # ODE-LSTM 的 Δt 需針對實際保留的序列重新計算；
        # 第一筆保留觀測設為 0，避免引用已被截斷的更早觀測。
        hist_dt = np.diff(hist_time, prepend=hist_time[0])
        hist_dt[0] = 0.0
        hist_dt = np.clip(hist_dt, 0.0, None)

        # 最後一筆歷史觀測至預測 target 的時間間隔。
        # 這是 one-step-ahead 預測時將 ODE 隱藏狀態推進至 target 時點所必需的資訊。
        target_dt = max(
            float(target["Time_in_Years"]) - float(hist_time[-1]),
            0.0,
        )

        valid_len = len(hist)
        mask_vec = np.concatenate([
            np.zeros(max_len - valid_len, dtype=float),
            np.ones(valid_len, dtype=float),
        ])

        seq_y = pad_left_1d(hist_y, max_len, pad_value=0.0)
        # 第二通道為明確 padding mask，避免 0 被誤解為真實平均值觀測。
        seq_values = np.column_stack([seq_y, mask_vec])
        dt_values = pad_left_1d(hist_dt, max_len, pad_value=0.0).reshape(-1, 1)

        X_seq.append(seq_values)
        X_dt.append(dt_values)
        X_target_dt.append([target_dt])
        X_exog.append(target[exog_cols].to_numpy(dtype=float))
        y.append(float(target[y_z_col]))
        mask.append(mask_vec)

        meta.append({
            ".imp": int(target[".imp"]),
            "row_id": int(target["row_id"]),
            "ID": str(target["ID"]),
            "Group": str(target["Group"]),
            "Time_in_Years": float(target["Time_in_Years"]),
            "Time_Category": str(target["Time_Category"]),
            "Time_Num": int(target["Time_Num"]),
            "outcome": outcome,
            "outcome_label": OUTCOME_LABELS[outcome],
            # 直接保留原始尺度觀測值作為正式評估真值。
            # 不可由 float32 標準化 target 再反轉換後充當 y_true。
            "target_value_original": float(target[outcome]),
            "target_originally_observed": bool(target[observed_col]),
            "target_row_position": int(target_pos + 1),
            "history_length_available": int(target_pos),
            "history_length_used": int(valid_len),
            "target_gap_from_last_history": float(target_dt),
            "prediction_scope": "last_originally_observed_target",
            "sequence_input": (
                "past standardized log outcome plus explicit padding-mask channel"
                if model_name != "ODE-LSTM"
                else (
                    "past standardized log outcome plus historical raw delta time "
                    "and last-history-to-target gap"
                )
            ),
        })

    if len(y) == 0:
        raise ValueError(
            f"split={split_name}, outcome={outcome}, model={model_name} "
            "沒有可建立的原始觀測 target。"
        )

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(X_dt, dtype=np.float32),
        np.asarray(X_target_dt, dtype=np.float32),
        np.asarray(X_exog, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        np.asarray(mask, dtype=np.float32),
        pd.DataFrame(meta),
    )


In [5]:
# ============================================================
# 4. 模型定義
# ============================================================

def build_lstm_narx(seq_shape, exog_shape):
    seq_in = Input(shape=seq_shape, name="past_log_outcome")
    exog_in = Input(shape=exog_shape, name="target_exogenous")

    x = LSTM(64, name="lstm_ar_branch")(seq_in)
    x = Dropout(0.20, name="lstm_dropout")(x)
    x = Concatenate(name="concat_ar_exog")([x, exog_in])
    out = Dense(1, name="output")(x)

    model = Model([seq_in, exog_in], out, name="LSTM_NARX")
    model.compile(optimizer="adam", loss="mse")
    return model


def build_tcn_model(seq_shape, exog_shape):
    seq_in = Input(shape=seq_shape, name="past_log_outcome")
    exog_in = Input(shape=exog_shape, name="target_exogenous")

    x = TCN(
        nb_filters=32,
        kernel_size=3,
        dilations=[1, 2, 4],
        dropout_rate=0.10,
        return_sequences=False,
        name="tcn_branch",
    )(seq_in)

    x = Concatenate(name="concat_tcn_exog")([x, exog_in])
    x = Dense(32, activation="relu", name="dense_hidden")(x)
    x = Dropout(0.20, name="dense_dropout")(x)
    out = Dense(1, name="output")(x)

    model = Model([seq_in, exog_in], out, name="TCN")
    model.compile(optimizer="adam", loss="mse")
    return model


class ODEFunc(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, h):
        return self.net(h)


class ODELSTMModel(nn.Module):
    """
    簡化式 ODE-LSTM：
    h(t+dt) = h(t) + dt * f(h(t))

    x_seq 第 1 通道為過去標準化對數 outcome，第 2 通道為 padding mask；
    x_dt 為保留歷史序列內重新計算的 raw Δt；
    x_target_dt 為最後一筆歷史觀測至 target 的 raw Δt；
    x_mask 僅控制 padding 位置是否更新，不作為模型特徵。
    """
    def __init__(self, exog_dim=4, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.odefunc = ODEFunc(hidden_dim)

        # x_seq 共有兩個通道：
        #   第 1 通道：標準化對數 outcome
        #   第 2 通道：padding indicator
        # ODE-LSTM 的 LSTMCell 只接收第 1 通道；padding 由 x_mask 另行控制。
        self.lstm_cell = nn.LSTMCell(input_size=1, hidden_size=hidden_dim)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim + exog_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq, x_dt, x_target_dt, x_mask, x_exog):
        if x_seq.ndim != 3 or x_seq.shape[2] < 1:
            raise ValueError(
                f"x_seq 應為三維張量且至少包含 outcome 通道，實際 shape={tuple(x_seq.shape)}"
            )

        batch_n, time_n, _ = x_seq.shape
        h = torch.zeros(batch_n, self.hidden_dim, device=x_seq.device)
        c = torch.zeros(batch_n, self.hidden_dim, device=x_seq.device)

        for t in range(time_n):
            x_t = x_seq[:, t, :1]
            dt_t = x_dt[:, t, 0].clamp(min=0.0, max=MAX_ODE_DT).unsqueeze(1)
            mask_t = x_mask[:, t].unsqueeze(1)

            h_ode = h + dt_t * self.odefunc(h)
            h_new, c_new = self.lstm_cell(x_t, (h_ode, c))

            h = mask_t * h_new + (1.0 - mask_t) * h
            c = mask_t * c_new + (1.0 - mask_t) * c

        # 歷史序列處理完成後，再將隱藏狀態由最後歷史觀測推進至 target 時點。
        dt_target = (
            x_target_dt[:, 0]
            .clamp(min=0.0, max=MAX_ODE_DT)
            .unsqueeze(1)
        )
        h_target = h + dt_target * self.odefunc(h)

        return self.fc(torch.cat([h_target, x_exog], dim=1)).squeeze(-1)


In [6]:
# ============================================================
# 5. 訓練、預測與單一模型執行函數
# ============================================================

def train_keras_model(model, X_train, y_train, X_val, y_val):
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        min_delta=0.0,
    )
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        shuffle=True,
        verbose=0,
    )
    return model, history.history


def train_odelstm_model(model, train_loader, val_loader):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE_TORCH)
    criterion = nn.MSELoss()

    best_state = None
    best_val = np.inf
    wait = 0
    history = {"loss": [], "val_loss": []}

    for _epoch in range(EPOCHS):
        model.train()
        train_losses = []

        for x_seq, x_dt, x_target_dt, x_mask, x_exog, y in train_loader:
            x_seq = x_seq.to(DEVICE)
            x_dt = x_dt.to(DEVICE)
            x_target_dt = x_target_dt.to(DEVICE)
            x_mask = x_mask.to(DEVICE)
            x_exog = x_exog.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            pred = model(x_seq, x_dt, x_target_dt, x_mask, x_exog)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.item()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_seq, x_dt, x_target_dt, x_mask, x_exog, y in val_loader:
                x_seq = x_seq.to(DEVICE)
                x_dt = x_dt.to(DEVICE)
                x_target_dt = x_target_dt.to(DEVICE)
                x_mask = x_mask.to(DEVICE)
                x_exog = x_exog.to(DEVICE)
                y = y.to(DEVICE)
                val_losses.append(
                    float(criterion(model(x_seq, x_dt, x_target_dt, x_mask, x_exog), y).item())
                )

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


def run_one_model_one_outcome(prediction_datasets, model_name: str, outcome: str):
    eval_all = []
    metrics_all = []
    loss_all = []
    sample_size_rows = []
    scaler_rows = []
    target_rows = []

    run_seed = model_run_seed(model_name, outcome)

    for dataset_id, df_dataset in prediction_datasets.items():
        print(f"  {model_name} | "f"{OUTCOME_LABELS[outcome]} | "f"prediction dataset {dataset_id}")

        # 同一 model/outcome 的每組插補資料重新使用相同初始化種子。
        set_all_seeds(run_seed)

        train_raw = df_dataset[df_dataset["split"] == "train"].copy()
        scaler = fit_scaler_from_train(train_raw, outcome)
        df_scaled = apply_scaler(df_dataset, scaler, outcome)

        Xs_tr, Xdt_tr, Xtargetdt_tr, Xe_tr, y_tr, mask_tr, meta_tr = build_sequence_dataset(
            df_scaled, "train", outcome, model_name
        )
        Xs_va, Xdt_va, Xtargetdt_va, Xe_va, y_va, mask_va, meta_va = build_sequence_dataset(
            df_scaled, "validation", outcome, model_name
        )
        Xs_te, Xdt_te, Xtargetdt_te, Xe_te, y_te, mask_te, meta_te = build_sequence_dataset(
            df_scaled, "test", outcome, model_name
        )

        for split_name, meta_df in [
            ("train", meta_tr), ("validation", meta_va), ("test", meta_te)
        ]:
            tmp = meta_df.copy()
            tmp["split"] = split_name
            tmp["model"] = model_name
            target_rows.append(tmp)

        sample_size_rows.append({
            ".imp": dataset_id,
            "model": model_name,
            "outcome": outcome,
            "outcome_label": OUTCOME_LABELS[outcome],
            "train_target_n": len(y_tr),
            "validation_target_n": len(y_va),
            "test_target_n": len(y_te),
            "train_patient_n": meta_tr["ID"].nunique(),
            "validation_patient_n": meta_va["ID"].nunique(),
            "test_patient_n": meta_te["ID"].nunique(),
            "run_seed": run_seed,
        })

        for variable, info in scaler.items():
            scaler_rows.append({
                ".imp": dataset_id,
                "model": model_name,
                "outcome": outcome,
                "variable": variable,
                "train_mean": info["mean"],
                "train_sd": info["sd"],
                "run_seed": run_seed,
            })

        if model_name == "LSTM-NARX":
            tf.keras.backend.clear_session()
            set_all_seeds(run_seed)
            model = build_lstm_narx(Xs_tr.shape[1:], Xe_tr.shape[1:])
            model, history = train_keras_model(
                model, [Xs_tr, Xe_tr], y_tr, [Xs_va, Xe_va], y_va
            )
            y_pred_std = model.predict([Xs_te, Xe_te], verbose=0).ravel()
            tf.keras.backend.clear_session()

        elif model_name == "TCN":
            tf.keras.backend.clear_session()
            set_all_seeds(run_seed)
            model = build_tcn_model(Xs_tr.shape[1:], Xe_tr.shape[1:])
            model, history = train_keras_model(
                model, [Xs_tr, Xe_tr], y_tr, [Xs_va, Xe_va], y_va
            )
            y_pred_std = model.predict([Xs_te, Xe_te], verbose=0).ravel()
            tf.keras.backend.clear_session()

        elif model_name == "ODE-LSTM":
            train_ds = TensorDataset(
                torch.tensor(Xs_tr, dtype=torch.float32),
                torch.tensor(Xdt_tr, dtype=torch.float32),
                torch.tensor(Xtargetdt_tr, dtype=torch.float32),
                torch.tensor(mask_tr, dtype=torch.float32),
                torch.tensor(Xe_tr, dtype=torch.float32),
                torch.tensor(y_tr, dtype=torch.float32),
            )
            val_ds = TensorDataset(
                torch.tensor(Xs_va, dtype=torch.float32),
                torch.tensor(Xdt_va, dtype=torch.float32),
                torch.tensor(Xtargetdt_va, dtype=torch.float32),
                torch.tensor(mask_va, dtype=torch.float32),
                torch.tensor(Xe_va, dtype=torch.float32),
                torch.tensor(y_va, dtype=torch.float32),
            )
            test_ds = TensorDataset(
                torch.tensor(Xs_te, dtype=torch.float32),
                torch.tensor(Xdt_te, dtype=torch.float32),
                torch.tensor(Xtargetdt_te, dtype=torch.float32),
                torch.tensor(mask_te, dtype=torch.float32),
                torch.tensor(Xe_te, dtype=torch.float32),
                torch.tensor(y_te, dtype=torch.float32),
            )

            generator = torch.Generator().manual_seed(run_seed)
            train_loader = DataLoader(
                train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=generator
            )
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
            test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

            set_all_seeds(run_seed)
            # Xs_tr 第 1 通道為 outcome，第 2 通道為 padding indicator。
            # ODELSTMModel 內部僅將 outcome 通道送入 LSTMCell，
            # padding 則由 mask_tr 顯式控制，因此 input_size 固定為 1。
            model = ODELSTMModel(
                exog_dim=Xe_tr.shape[1],
                hidden_dim=64,
            ).to(DEVICE)

            model, history = train_odelstm_model(model, train_loader, val_loader)

            model.eval()
            preds = []
            with torch.no_grad():
                for x_seq, x_dt, x_target_dt, x_mask, x_exog, _ in test_loader:
                    pred = model(
                        x_seq.to(DEVICE),
                        x_dt.to(DEVICE),
                        x_target_dt.to(DEVICE),
                        x_mask.to(DEVICE),
                        x_exog.to(DEVICE),
                    )
                    preds.append(pred.cpu().numpy())
            y_pred_std = np.concatenate(preds).reshape(-1)

        else:
            raise ValueError(f"未知模型：{model_name}")

        # 正式測試真值直接取自原始尺度的實際觀測值。
        # y_te 是 float32 的標準化對數 target；若先反轉換再當真值，
        # 不同插補資料使用不同 train scaler 時會產生約 1e-5 等級的
        # 數值誤差，造成「真值跨插補不一致」的偽警報。
        y_true = meta_te["target_value_original"].to_numpy(dtype=float)
        y_pred = inverse_transform_y(y_pred_std, scaler, outcome)

        # 僅作數值轉換合理性檢查，不把反轉換值當成正式真值。
        y_true_from_scaled = inverse_transform_y(
            np.asarray(y_te, dtype=np.float64),
            scaler,
            outcome,
        )
        if not np.allclose(
            y_true,
            y_true_from_scaled,
            rtol=1e-5,
            atol=1e-6,
        ):
            raise RuntimeError(
                f"{model_name} / {outcome} / prediction dataset {dataset_id}："
                "標準化 target 反轉換後與原始觀測值差異過大。"
            )

        pred_df = meta_te.copy()
        pred_df["model"] = model_name
        pred_df["run_seed"] = run_seed
        pred_df["y_true"] = y_true
        pred_df["y_pred"] = y_pred
        pred_df["error"] = pred_df["y_pred"] - pred_df["y_true"]
        pred_df["abs_error"] = np.abs(pred_df["error"])
        pred_df["ape"] = np.abs(pred_df["error"] / pred_df["y_true"]) * 100

        if not pred_df["target_originally_observed"].all():
            raise RuntimeError("測試 target 出現非原始觀測值。")

        eval_all.append(pred_df)
        metrics_all.append(metrics_by_group(pred_df))

        hist_df = pd.DataFrame(history)
        hist_df["epoch"] = np.arange(1, len(hist_df) + 1)
        hist_df[".imp"] = dataset_id
        hist_df["model"] = model_name
        hist_df["outcome"] = outcome
        hist_df["outcome_label"] = OUTCOME_LABELS[outcome]
        hist_df["run_seed"] = run_seed
        loss_all.append(hist_df)

    return {
        "eval_df": pd.concat(eval_all, ignore_index=True),
        "metrics_by_imp_group": pd.concat(metrics_all, ignore_index=True),
        "loss_history": pd.concat(loss_all, ignore_index=True),
        "sequence_sample_size": pd.DataFrame(sample_size_rows),
        "scaler_parameters": pd.DataFrame(scaler_rows),
        "target_manifest": pd.concat(target_rows, ignore_index=True),
    }


In [7]:
# ============================================================
# 6. 執行三種 DL 模型
# ============================================================

prediction_datasets = {
    int(dataset_id): dataset_df.sort_values(
        ["ID", "Time_in_Years", "row_id"]
    ).reset_index(drop=True)
    for dataset_id, dataset_df in dt_all.groupby(
        ".imp",
        sort=True,
    )
}

all_eval = []
all_metrics = []
all_loss = []
all_sample_size = []
all_scalers = []
all_targets = []

for outcome in OUTCOMES:
    print("\n====================================================")
    print("Outcome:", OUTCOME_LABELS[outcome])
    print("====================================================")

    for model_name in MODEL_ORDER:
        result = run_one_model_one_outcome(
            prediction_datasets=prediction_datasets,
            model_name=model_name,
            outcome=outcome
        )

        all_eval.append(
            result["eval_df"]
        )

        all_metrics.append(
            result["metrics_by_imp_group"]
        )

        all_loss.append(
            result["loss_history"]
        )

        all_sample_size.append(
            result["sequence_sample_size"]
        )

        all_scalers.append(
            result["scaler_parameters"]
        )

        all_targets.append(
            result["target_manifest"]
        )

# ============================================================
# 6.1 合併所有模型結果
# ============================================================

if len(all_eval) == 0:
    raise RuntimeError("未產生任何 DL 預測結果。")

if len(all_metrics) == 0:
    raise RuntimeError("未產生任何 DL 預測指標。")

if len(all_loss) == 0:
    raise RuntimeError("未產生任何訓練損失紀錄。")

if len(all_sample_size) == 0:
    raise RuntimeError("未產生任何序列樣本數紀錄。")

if len(all_scalers) == 0:
    raise RuntimeError("未產生任何標準化參數紀錄。")

if len(all_targets) == 0:
    raise RuntimeError("未產生任何 target manifest。")


dl_test_predictions_individual = pd.concat(
    all_eval,
    ignore_index=True,
)

dl_prediction_metrics_raw = pd.concat(
    all_metrics,
    ignore_index=True,
)

dl_loss_history_by_epoch = pd.concat(
    all_loss,
    ignore_index=True,
)

dl_sequence_sample_size = pd.concat(
    all_sample_size,
    ignore_index=True,
)

dl_scaler_parameters = pd.concat(
    all_scalers,
    ignore_index=True,
)

dl_target_manifest = pd.concat(
    all_targets,
    ignore_index=True,
)


# ============================================================
# 6.2 基本完整性 QC
# ============================================================

expected_model_outcome_n = (
    len(MODEL_ORDER) *
    len(OUTCOMES)
)

actual_model_outcome_n = (
    dl_test_predictions_individual[
        ["model", "outcome"]
    ]
    .drop_duplicates()
    .shape[0]
)

if actual_model_outcome_n != expected_model_outcome_n:
    raise RuntimeError(
        "模型與 outcome 組合數不完整："
        f"預期 {expected_model_outcome_n} 組，"
        f"實際 {actual_model_outcome_n} 組。"
    )

if dl_test_predictions_individual.empty:
    raise RuntimeError(
        "dl_test_predictions_individual 為空。"
    )

if dl_test_predictions_individual[
    "y_true"
].isna().any():
    raise RuntimeError(
        "測試真值 y_true 含缺失值。"
    )

if dl_test_predictions_individual[
    "y_pred"
].isna().any():
    raise RuntimeError(
        "模型預測值 y_pred 含缺失值。"
    )

if not np.isfinite(
    dl_test_predictions_individual[
        ["y_true", "y_pred"]
    ].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "測試真值或模型預測值含非有限值。"
    )


# ============================================================
# 6.3 Target 一致性 QC
# ============================================================

test_targets = (
    dl_target_manifest.loc[
        dl_target_manifest["split"].eq("test"),
        [
            ".imp",
            "model",
            "outcome",
            "row_id",
            "ID",
            "Group",
        ],
    ]
    .drop_duplicates()
)

target_model_consistency = (
    test_targets
    .groupby(
        [
            ".imp",
            "outcome",
            "row_id",
            "ID",
            "Group",
        ]
    )["model"]
    .nunique()
    .reset_index(
        name="n_models"
    )
)

if not target_model_consistency[
    "n_models"
].eq(
    len(MODEL_ORDER)
).all():
    bad_target_coverage = (
        target_model_consistency.loc[
            ~target_model_consistency[
                "n_models"
            ].eq(len(MODEL_ORDER))
        ]
        .head(20)
    )

    raise RuntimeError(
        "三種 DL 模型的測試 target 不一致：\n"
        + bad_target_coverage.to_string(
            index=False
        )
    )


truth_consistency = (
    dl_test_predictions_individual
    .groupby(
        [
            "outcome",
            "row_id",
            "ID",
            "Group",
        ],
        as_index=False,
    )
    .agg(
        y_true_min=(
            "y_true",
            "min",
        ),
        y_true_max=(
            "y_true",
            "max",
        ),
        dataset_n=(
            ".imp",
            "nunique",
        ),
        n_models=(
            "model",
            "nunique",
        ),
    )
)

truth_consistency[
    "truth_range"
] = (
    truth_consistency["y_true_max"]
    - truth_consistency["y_true_min"]
)

truth_consistency[
    "passed"
] = (
    truth_consistency[
        "truth_range"
    ].abs().le(1e-12)
    & truth_consistency[
        "dataset_n"
    ].eq(EXPECTED_M)
    & truth_consistency[
        "n_models"
    ].eq(len(MODEL_ORDER))
)

if not truth_consistency[
    "passed"
].all():
    bad_truth = (
        truth_consistency.loc[
            ~truth_consistency[
                "passed"
            ]
        ]
        .head(20)
    )

    raise RuntimeError(
        "原始觀測測試真值、prediction dataset 數量"
        "或模型覆蓋不一致：\n"
        + bad_truth.to_string(
            index=False
        )
    )


# ============================================================
# 6.4 建立最終分組預測指標表
# ============================================================

dl_prediction_metrics_summary_group = (
    summarize_single_dataset_metrics(
        dl_prediction_metrics_raw
    )
)

required_metric_cols = {
    "outcome",
    "model",
    "Group_eval",
    "RMSE",
    "MAE",
    "MAPE",
    "Bias",
    "n_test",
}

missing_metric_cols = (
    required_metric_cols
    - set(
        dl_prediction_metrics_summary_group.columns
    )
)

if missing_metric_cols:
    raise RuntimeError(
        "預測指標總表缺少欄位："
        + ", ".join(
            sorted(missing_metric_cols)
        )
    )

expected_metric_row_n = (
    len(OUTCOMES)
    * len(MODEL_ORDER)
    * 3
)

if (
    len(
        dl_prediction_metrics_summary_group
    )
    != expected_metric_row_n
):
    raise RuntimeError(
        "預測指標總表列數不正確："
        f"預期 {expected_metric_row_n} 列，"
        f"實際 "
        f"{len(dl_prediction_metrics_summary_group)} 列。"
    )


# ============================================================
# 6.5 ODE-LSTM target-gap QC
# ============================================================

ode_target_gap_qc = (
    dl_target_manifest.loc[
        dl_target_manifest[
            "model"
        ].eq("ODE-LSTM"),
        [
            ".imp",
            "outcome",
            "split",
            "row_id",
            "target_gap_from_last_history",
        ],
    ]
    .copy()
)

if ode_target_gap_qc.empty:
    raise RuntimeError(
        "找不到 ODE-LSTM target gap 紀錄。"
    )

ode_gap_values = (
    ode_target_gap_qc[
        "target_gap_from_last_history"
    ]
    .to_numpy(dtype=float)
)

if (
    ode_target_gap_qc[
        "target_gap_from_last_history"
    ].isna().any()
    or not np.isfinite(
        ode_gap_values
    ).all()
    or (
        ode_target_gap_qc[
            "target_gap_from_last_history"
        ] < 0
    ).any()
):
    raise RuntimeError(
        "ODE-LSTM target gap 含缺失、"
        "非有限值或負值。"
    )

ode_target_gap_summary = (
    ode_target_gap_qc
    .groupby(
        [
            "outcome",
            "split",
        ],
        as_index=False,
    )
    .agg(
        n_target=(
            "row_id",
            "size",
        ),
        gap_min=(
            "target_gap_from_last_history",
            "min",
        ),
        gap_mean=(
            "target_gap_from_last_history",
            "mean",
        ),
        gap_median=(
            "target_gap_from_last_history",
            "median",
        ),
        gap_max=(
            "target_gap_from_last_history",
            "max",
        ),
    )
)


# ============================================================
# 6.6 MAPE 分母穩定性 QC
# ============================================================
# 每個測試 target 在 individual prediction 表中會因三個模型
# 重複出現三次，因此先建立不重複的測試真值資料，再計算 QC。

unique_test_truth = (
    dl_test_predictions_individual[
        [
            "outcome",
            "outcome_label",
            "row_id",
            "ID",
            "Group",
            "y_true",
        ]
    ]
    .drop_duplicates(
        subset=[
            "outcome",
            "row_id",
            "ID",
            "Group",
        ]
    )
    .reset_index(drop=True)
)

if unique_test_truth.empty:
    raise RuntimeError(
        "無法建立不重複的測試真值資料。"
    )

mape_denominator_qc = (
    unique_test_truth
    .groupby(
        [
            "outcome",
            "outcome_label",
        ],
        as_index=False,
    )
    .agg(
        n_test=(
            "y_true",
            "size",
        ),
        min_y_true=(
            "y_true",
            "min",
        ),
        q01_y_true=(
            "y_true",
            lambda x: float(
                x.quantile(0.01)
            ),
        ),
        near_zero_n=(
            "y_true",
            lambda x: int(
                (
                    np.abs(x) <= 1e-6
                ).sum()
            ),
        ),
    )
)

if not mape_denominator_qc[
    "near_zero_n"
].eq(0).all():
    raise RuntimeError(
        "測試真值存在接近 0 的資料，"
        "MAPE 可能不穩定；"
        "請檢查 dl_mape_denominator_qc.csv。"
    )

expected_test_n = {
    "HbA1c": 271,
    "Glu_AC": 146,
    "LDL": 252,
    "TG": 252,
}

for outcome_name, expected_n in (
    expected_test_n.items()
):
    matched_row = (
        mape_denominator_qc.loc[
            mape_denominator_qc[
                "outcome"
            ].eq(outcome_name)
        ]
    )

    if len(matched_row) != 1:
        raise RuntimeError(
            f"{outcome_name} 的 MAPE QC "
            "找不到唯一一列結果。"
        )

    actual_n = int(
        matched_row[
            "n_test"
        ].iloc[0]
    )

    if actual_n != expected_n:
        raise RuntimeError(
            f"{outcome_name} 的測試 target 數量"
            f"不一致：預期 {expected_n}，"
            f"實際 {actual_n}。"
        )


# ============================================================
# 6.7 輸出 CSV
# ============================================================

dl_prediction_metrics_summary_group.to_csv(
    OUT_DIR
    / "dl_prediction_metrics_summary_group.csv",
    index=False,
    encoding="utf-8-sig",
)

dl_prediction_metrics_summary_group.to_csv(
    TABLE_DIR
    / "dl_prediction_metrics_summary_group.csv",
    index=False,
    encoding="utf-8-sig",
)

dl_test_predictions_individual.to_csv(
    TABLE_DIR
    / "dl_test_predictions_individual.csv",
    index=False,
    encoding="utf-8-sig",
)

dl_loss_history_by_epoch.to_csv(
    TABLE_DIR
    / "dl_loss_history_by_epoch.csv",
    index=False,
    encoding="utf-8-sig",
)

dl_sequence_sample_size.to_csv(
    TABLE_DIR
    / "dl_sequence_sample_size.csv",
    index=False,
    encoding="utf-8-sig",
)

dl_scaler_parameters.to_csv(
    TABLE_DIR
    / "dl_scaler_parameters.csv",
    index=False,
    encoding="utf-8-sig",
)

dl_target_manifest.to_csv(
    TABLE_DIR
    / "dl_target_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

truth_consistency.to_csv(
    TABLE_DIR
    / "dl_target_truth_consistency_qc.csv",
    index=False,
    encoding="utf-8-sig",
)

ode_target_gap_summary.to_csv(
    TABLE_DIR
    / "dl_ode_target_gap_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

mape_denominator_qc.to_csv(
    TABLE_DIR
    / "dl_mape_denominator_qc.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 6.8 模型設定輸出
# ============================================================

model_settings = pd.DataFrame(
    [
        {
            "setting": "SCRIPT_VERSION",
            "value": SCRIPT_VERSION,
        },
        {
            "setting": "OUTCOMES",
            "value": ", ".join(OUTCOMES),
        },
        {
            "setting": "MODEL_ORDER",
            "value": ", ".join(MODEL_ORDER),
        },
        {
            "setting": "MAX_LEN",
            "value": MAX_LEN,
        },
        {
            "setting": "EPOCHS",
            "value": EPOCHS,
        },
        {
            "setting": "BATCH_SIZE",
            "value": BATCH_SIZE,
        },
        {
            "setting": "PATIENCE",
            "value": PATIENCE,
        },
        {
            "setting": "SEED",
            "value": SEED,
        },
        {
            "setting": "prediction_scope",
            "value": (
                "last originally observed target "
                "per evaluable patient"
            ),
        },
        {
            "setting": "history_values",
            "value": (
                "originally observed historical outcomes only; "
                "no outcome imputation"
            ),
        },
        {
            "setting": "target_values",
            "value": (
                "exact originally observed values "
                "on the original scale"
            ),
        },
        {
            "setting": "LSTM_TCN_sequence",
            "value": (
                "past standardized log outcome plus "
                "explicit padding-mask channel"
            ),
        },
        {
            "setting": "LSTM_TCN_padding",
            "value": (
                "left zero padding to length 7 "
                "with explicit mask channel"
            ),
        },
        {
            "setting": "ODE_sequence",
            "value": (
                "past standardized log outcome, "
                "historical raw delta time, and "
                "last-history-to-target gap"
            ),
        },
        {
            "setting": "ODE_padding",
            "value": (
                "left zero padding with structural update mask"
            ),
        },
        {
            "setting": "ODE_exogenous",
            "value": (
                "standardized age, sex, standardized BMI, "
                "statin indicator"
            ),
        },
        {
            "setting": "metric_scale",
            "value": "original scale",
        },
        {
            "setting": "metric_groups",
            "value": "Overall, Control, Statin",
        },
        {
            "setting": "metric_uncertainty",
            "value": (
                "patient-level bootstrap 95% CI calculated "
                "later in R from individual predictions"
            ),
        },
        {
            "setting": "upstream_data_role",
            "value": (
                " | ".join(data_role_values)
                if data_role_values
                else "data_role column not supplied"
            ),
        },
        {
            "setting": "prediction_preprocessing",
            "value": (
                "patient split fixed first; outcomes not imputed; "
                "BMI filled from train-only medians upstream"
            ),
        },
        {
            "setting": "DEVICE",
            "value": DEVICE,
        },
    ]
)

model_settings.to_csv(
    TABLE_DIR
    / "dl_model_settings.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 6.9 Excel 輸出
# ============================================================

try:
    with pd.ExcelWriter(
        TABLE_DIR
        / "DL_model_outputs_type2.xlsx",
        engine="openpyxl",
    ) as writer:

        dl_prediction_metrics_summary_group.to_excel(
            writer,
            sheet_name="metrics_summary",
            index=False,
        )

        dl_sequence_sample_size.to_excel(
            writer,
            sheet_name="sequence_sample_size",
            index=False,
        )

        dl_scaler_parameters.to_excel(
            writer,
            sheet_name="scaler_parameters",
            index=False,
        )

        dl_target_manifest.to_excel(
            writer,
            sheet_name="target_manifest",
            index=False,
        )

        truth_consistency.to_excel(
            writer,
            sheet_name="target_truth_qc",
            index=False,
        )

        ode_target_gap_summary.to_excel(
            writer,
            sheet_name="ode_target_gap",
            index=False,
        )

        mape_denominator_qc.to_excel(
            writer,
            sheet_name="mape_denominator_qc",
            index=False,
        )

        input_qc.to_excel(
            writer,
            sheet_name="input_qc",
            index=False,
        )

        model_settings.to_excel(
            writer,
            sheet_name="model_settings",
            index=False,
        )

except Exception as exc:
    warnings.warn(
        "Excel 輸出失敗；CSV 已保留。"
        f"原因：{exc}"
    )


print("DL 預測指標已完成。")
print(
    dl_prediction_metrics_summary_group.head(20)
)

print("\nMAPE 分母 QC：")
print(
    mape_denominator_qc
)



Outcome: HbA1c
  LSTM-NARX | HbA1c | prediction dataset 1

  TCN | HbA1c | prediction dataset 1
  ODE-LSTM | HbA1c | prediction dataset 1

Outcome: Glu-AC
  LSTM-NARX | Glu-AC | prediction dataset 1
  TCN | Glu-AC | prediction dataset 1
  ODE-LSTM | Glu-AC | prediction dataset 1

Outcome: LDL
  LSTM-NARX | LDL | prediction dataset 1
  TCN | LDL | prediction dataset 1
  ODE-LSTM | LDL | prediction dataset 1

Outcome: TG
  LSTM-NARX | TG | prediction dataset 1
  TCN | TG | prediction dataset 1
  ODE-LSTM | TG | prediction dataset 1
DL 預測指標已完成。
   outcome outcome_label      model Group_eval    Group_label       RMSE  \
0    HbA1c         HbA1c  LSTM-NARX    Overall        Overall   0.844261   
1    HbA1c         HbA1c  LSTM-NARX    Control  Control group   0.878562   
2    HbA1c         HbA1c  LSTM-NARX     Statin   Statin group   0.814657   
3    HbA1c         HbA1c        TCN    Overall        Overall   0.866745   
4    HbA1c         HbA1c        TCN    Control  Control group   0.86759

In [8]:
# ============================================================
# 7. 中英文圖形與圖形用樣本數表
# ============================================================

def prepare_loss_plot_df(
    loss_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    整理各模型與結果變項的訓練及驗證損失。

    方案 A 僅使用單一 prediction-safe dataset。
    不同 model-outcome 組合可能因 early stopping
    而具有不同訓練回合數。
    """

    required_cols = {
        ".imp",
        "model",
        "outcome",
        "outcome_label",
        "epoch",
        "loss",
        "val_loss",
    }

    missing_cols = required_cols.difference(
        loss_df.columns
    )

    if missing_cols:
        raise ValueError(
            f"loss_df 缺少必要欄位："
            f"{sorted(missing_cols)}"
        )

    long_df = loss_df.melt(
        id_vars=[
            ".imp",
            "model",
            "outcome",
            "outcome_label",
            "epoch",
        ],
        value_vars=[
            "loss",
            "val_loss",
        ],
        var_name="loss_type",
        value_name="loss_value",
    )

    long_df["epoch"] = pd.to_numeric(
        long_df["epoch"],
        errors="coerce",
    )

    long_df["loss_value"] = pd.to_numeric(
        long_df["loss_value"],
        errors="coerce",
    )

    long_df = long_df.dropna(
        subset=[
            "model",
            "outcome",
            "epoch",
            "loss_type",
            "loss_value",
        ]
    )

    return (
        long_df
        .groupby(
            [
                "model",
                "outcome",
                "outcome_label",
                "epoch",
                "loss_type",
            ],
            dropna=False,
        )
        .agg(
            loss_mean=(
                "loss_value",
                "mean",
            ),
            loss_sd=(
                "loss_value",
                "std",
            ),
            dataset_n=(
                ".imp",
                "nunique",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "model",
                "outcome",
                "loss_type",
                "epoch",
            ]
        )
        .reset_index(drop=True)
    )


def plot_loss_curves_12panel(
    loss_df: pd.DataFrame,
    lang: str,
):
    """
    繪製三種模型與四項結果變項的
    訓練及驗證損失曲線。
    """

    if lang not in {
        "zh",
        "en",
    }:
        raise ValueError(
            "lang 必須為 'zh' 或 'en'。"
        )

    plot_df = prepare_loss_plot_df(
        loss_df
    )

    cfg = {
        "zh": {
            "train": "訓練損失",
            "val": "驗證損失",
            "epoch": "訓練回合",
            "loss": "標準化對數尺度 MSE",
            "title": (
                "三種深度學習模型之"
                "訓練與驗證損失曲線"
            ),
            "note": (
                "註：損失為標準化自然對數尺度之"
                "均方誤差（MSE）；曲線呈現單一 "
                "prediction-safe dataset 的訓練紀錄。"
                "各模型與結果變項可能因早期停止而"
                "具有不同訓練回合數。"
            ),
            "suffix": "zh",
        },
        "en": {
            "train": "Training loss",
            "val": "Validation loss",
            "epoch": "Epoch",
            "loss": (
                "MSE on standardized log scale"
            ),
            "title": (
                "Training and validation loss curves "
                "for three deep-learning models"
            ),
            "note": (
                "Note: Loss is MSE on the standardized "
                "natural-log scale. The curves show the "
                "training history from a single "
                "prediction-safe dataset. Training may "
                "terminate at different epochs across "
                "model-outcome combinations because of "
                "early stopping."
            ),
            "suffix": "en",
        },
    }[lang]

    fig, axes = plt.subplots(
        len(MODEL_ORDER),
        len(OUTCOMES),
        figsize=(18, 10.5),
        sharex=False,
        sharey=False,
    )

    for row_index, model_name in enumerate(
        MODEL_ORDER
    ):
        for col_index, outcome in enumerate(
            OUTCOMES
        ):
            ax = axes[
                row_index,
                col_index,
            ]

            sub = plot_df[
                (
                    plot_df["model"]
                    == model_name
                )
                & (
                    plot_df["outcome"]
                    == outcome
                )
            ].copy()

            for (
                loss_type,
                line_style,
                line_label,
            ) in [
                (
                    "loss",
                    "-",
                    cfg["train"],
                ),
                (
                    "val_loss",
                    "--",
                    cfg["val"],
                ),
            ]:
                ss = (
                    sub[
                        sub["loss_type"]
                        == loss_type
                    ]
                    .sort_values("epoch")
                    .copy()
                )

                if ss.empty:
                    continue

                ax.plot(
                    ss["epoch"],
                    ss["loss_mean"],
                    linestyle=line_style,
                    linewidth=1.2,
                    label=line_label,
                )

            ax.set_title(
                (
                    f"{model_name}: "
                    f"{OUTCOME_LABELS[outcome]}"
                ),
                fontsize=10,
                fontweight="bold",
            )

            ax.set_xlabel(
                cfg["epoch"]
            )

            ax.set_ylabel(
                cfg["loss"]
            )

            ax.grid(
                alpha=0.25
            )

    handles, labels = (
        axes[0, 0]
        .get_legend_handles_labels()
    )

    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(
            0.5,
            0.055,
        ),
        ncol=2,
        frameon=False,
    )

    fig.suptitle(
        cfg["title"],
        fontsize=16,
        fontweight="bold",
    )

    fig.text(
        0.5,
        0.018,
        cfg["note"],
        ha="center",
        va="bottom",
        fontsize=8,
        wrap=True,
    )

    fig.tight_layout(
        rect=[
            0,
            0.10,
            1,
            0.94,
        ]
    )

    out_file = (
        FIGURE_DIR
        / (
            "dl_loss_curves_"
            f"12panel_{cfg['suffix']}.png"
        )
    )

    fig.savefig(
        out_file,
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none",
    )

    plt.close(fig)

    return out_file


def time_category_to_num(
    value,
):
    """
    將 T1-T19 轉換為整數 1-19。
    """

    if pd.isna(value):
        return np.nan

    try:
        return int(
            str(value).replace(
                "T",
                "",
            )
        )
    except (
        TypeError,
        ValueError,
    ):
        return np.nan


def prepare_prediction_trajectory_df(
    pred_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    建立區間別實際值與預測值摘要。

    每一 model-outcome-group-time cell 計算：
    - true_mean：實際 target 平均值；
    - pred_mean：預測 target 平均值；
    - true_sd / pred_sd：跨病患標準差；
    - true_se / pred_se：平均值標準誤；
    - true_ci_lower / true_ci_upper：
      實際平均值之描述性 95% CI；
    - pred_ci_lower / pred_ci_upper：
      預測平均值之描述性 95% CI；
    - Time_mid：該區間平均追蹤時間；
    - n_target：唯一測試 target 數。

    圖形僅繪製平均值點，不繪製連線或 CI。
    SD、SE 與 95% CI 保留於輸出 CSV，
    供附錄或資料品質檢查使用。
    """

    required_cols = {
        ".imp",
        "model",
        "outcome",
        "outcome_label",
        "Group",
        "Time_Category",
        "row_id",
        "ID",
        "y_true",
        "y_pred",
        "Time_in_Years",
    }

    missing_cols = required_cols.difference(
        pred_df.columns
    )

    if missing_cols:
        raise ValueError(
            f"pred_df 缺少必要欄位："
            f"{sorted(missing_cols)}"
        )

    df = pred_df.copy()

    df["y_true"] = pd.to_numeric(
        df["y_true"],
        errors="coerce",
    )

    df["y_pred"] = pd.to_numeric(
        df["y_pred"],
        errors="coerce",
    )

    df["Time_in_Years"] = pd.to_numeric(
        df["Time_in_Years"],
        errors="coerce",
    )

    df["Time_Category"] = pd.Categorical(
        df["Time_Category"],
        categories=[
            f"T{i}"
            for i in range(
                1,
                20,
            )
        ],
        ordered=True,
    )

    target_level = (
        df.dropna(
            subset=[
                "model",
                "outcome",
                "Group",
                "Time_Category",
                "row_id",
                "ID",
                "y_true",
                "y_pred",
                "Time_in_Years",
            ]
        )
        .groupby(
            [
                "model",
                "outcome",
                "outcome_label",
                "Group",
                "Time_Category",
                "row_id",
                "ID",
            ],
            observed=True,
            dropna=False,
        )
        .agg(
            y_true=(
                "y_true",
                "first",
            ),
            y_pred=(
                "y_pred",
                "mean",
            ),
            Time_in_Years=(
                "Time_in_Years",
                "first",
            ),
            dataset_n=(
                ".imp",
                "nunique",
            ),
        )
        .reset_index()
    )

    if not target_level[
        "dataset_n"
    ].eq(EXPECTED_M).all():
        raise RuntimeError(
            "圖形彙整時發現部分 target "
            "未包含預期的 prediction-safe "
            "dataset 預測。"
        )

    duplicate_target = (
        target_level
        .duplicated(
            subset=[
                "model",
                "outcome",
                "row_id",
            ],
            keep=False,
        )
    )

    if duplicate_target.any():
        duplicated_rows = (
            target_level.loc[
                duplicate_target,
                [
                    "model",
                    "outcome",
                    "row_id",
                    "ID",
                    "Time_Category",
                ],
            ]
            .head(20)
        )

        raise RuntimeError(
            "圖形彙整時發現重複 target：\n"
            f"{duplicated_rows}"
        )

    out = (
        target_level
        .groupby(
            [
                "model",
                "outcome",
                "outcome_label",
                "Group",
                "Time_Category",
            ],
            observed=True,
            dropna=False,
        )
        .agg(
            true_mean=(
                "y_true",
                "mean",
            ),
            true_sd=(
                "y_true",
                "std",
            ),
            pred_mean=(
                "y_pred",
                "mean",
            ),
            pred_sd=(
                "y_pred",
                "std",
            ),
            Time_mid=(
                "Time_in_Years",
                "mean",
            ),
            n_target=(
                "row_id",
                "nunique",
            ),
        )
        .reset_index()
    )

    out["true_se"] = (
        out["true_sd"]
        / np.sqrt(
            out["n_target"]
        )
    )

    out["pred_se"] = (
        out["pred_sd"]
        / np.sqrt(
            out["n_target"]
        )
    )

    out["true_ci_lower"] = (
        out["true_mean"]
        - 1.96
        * out["true_se"]
    )

    out["true_ci_upper"] = (
        out["true_mean"]
        + 1.96
        * out["true_se"]
    )

    out["pred_ci_lower"] = (
        out["pred_mean"]
        - 1.96
        * out["pred_se"]
    )

    out["pred_ci_upper"] = (
        out["pred_mean"]
        + 1.96
        * out["pred_se"]
    )

    out["Time_Num"] = (
        out["Time_Category"]
        .astype(str)
        .map(
            time_category_to_num
        )
    )

    return (
        out.query(
            "n_target > 0"
        )
        .dropna(
            subset=[
                "Time_Num",
                "true_mean",
                "pred_mean",
                "Time_mid",
            ]
        )
        .sort_values(
            [
                "model",
                "outcome",
                "Group",
                "Time_Num",
            ]
        )
        .reset_index(
            drop=True
        )
    )


trajectory_interval_counts = (
    prepare_prediction_trajectory_df(
        dl_test_predictions_individual
    )
)

trajectory_interval_counts.to_csv(
    (
        TABLE_DIR
        / (
            "dl_trajectory_interval_"
            "counts_and_means.csv"
        )
    ),
    index=False,
    encoding="utf-8-sig",
)


def plot_prediction_points_12panel(
    pred_df: pd.DataFrame,
    x_var: str,
    lang: str,
):
    """
    繪製各模型、結果變項與組別之
    區間別實際平均值與預測平均值。

    僅繪製點，不連線，以避免將不同區間
    的不同病患錯誤解讀為連續縱向軌跡。

    x_var：
    - "Time_Num"：
      使用 T1-T19 為 x 軸；
    - "Time_mid"：
      使用各區間平均追蹤時間為 x 軸。
    """

    if x_var not in {
        "Time_Num",
        "Time_mid",
    }:
        raise ValueError(
            "x_var 必須為 "
            "'Time_Num' 或 'Time_mid'。"
        )

    if lang not in {
        "zh",
        "en",
    }:
        raise ValueError(
            "lang 必須為 'zh' 或 'en'。"
        )

    plot_df = (
        prepare_prediction_trajectory_df(
            pred_df
        )
        .loc[
            lambda x:
            x["n_target"]
            >= MIN_PLOT_TARGET_N
        ]
        .copy()
    )

    if lang == "zh":
        group_labels = (
            GROUP_LABELS_ZH
        )

        observed_label = (
            "測試集實際平均值"
        )

        predicted_label = (
            "模型預測平均值"
        )

        xlabel_period = (
            "追蹤時間區間"
        )

        xlabel_year = (
            "各區間平均追蹤時間（年）"
        )

        title_period = (
            "測試集最後一次原始觀測目標之"
            "區間別實際與預測平均值"
        )

        title_year = (
            "測試集最後一次原始觀測目標之"
            "實際與預測平均值"
            "（依各區間平均追蹤時間呈現）"
        )

        note_period = (
            "註：每位病患僅貢獻一筆最後一次"
            "原始觀測目標；每一點為該區間之"
            "跨病患平均值。"
            f"每組每區間 n < {MIN_PLOT_TARGET_N} "
            "者不繪示；圖中不連線，避免將不同"
            "區間之病患平均值誤解為個體縱向軌跡。"
            "各區間樣本數、SD、SE 與描述性 95% "
            "信賴區間請見 "
            "dl_trajectory_interval_counts_and_means.csv。"
        )

        note_year = (
            "註：各 T1–T19 區間平均值放置於該"
            "區間的平均追蹤時間，並非連續時間"
            "模型擬合曲線。每位病患僅貢獻一筆"
            "最後一次原始觀測目標；"
            f"每組每區間 n < {MIN_PLOT_TARGET_N} "
            "者不繪示，圖中不連線。"
        )

        suffix = "zh"

    else:
        group_labels = (
            GROUP_LABELS_EN
        )

        observed_label = (
            "Observed mean"
        )

        predicted_label = (
            "Predicted mean"
        )

        xlabel_period = (
            "Follow-up interval"
        )

        xlabel_year = (
            "Mean follow-up time "
            "within interval (years)"
        )

        title_period = (
            "Interval-specific observed and predicted "
            "means for the last originally observed "
            "test targets"
        )

        title_year = (
            "Observed and predicted means plotted at "
            "the mean follow-up time within each interval"
        )

        note_period = (
            "Note: Each patient contributes one last "
            "originally observed target, and each point "
            "represents a between-patient interval mean. "
            f"Cells with n < {MIN_PLOT_TARGET_N} are "
            "omitted. Points are not connected because "
            "different intervals contain different "
            "patients and should not be interpreted as "
            "individual longitudinal trajectories. "
            "Sample sizes, SDs, SEs, and descriptive 95% "
            "confidence intervals are reported in "
            "dl_trajectory_interval_counts_and_means.csv."
        )

        note_year = (
            "Note: T1-T19 interval means are positioned "
            "at the mean follow-up time within each "
            "interval. This is not a fitted continuous-time "
            "prediction curve. Each patient contributes "
            "one last originally observed target; "
            f"cells with n < {MIN_PLOT_TARGET_N} are "
            "omitted, and points are not connected."
        )

        suffix = "en"

    fig, axes = plt.subplots(
        len(MODEL_ORDER),
        len(OUTCOMES),
        figsize=(
            18,
            10.5,
        ),
        sharex=False,
        sharey=False,
    )

    colors = {
        "Control": "#F8766D",
        "Statin": "#00BFC4",
    }

    observed_marker = "o"
    predicted_marker = "s"

    # 讓同一組 observed 與 predicted 點稍微左右錯開，
    # 避免兩者完全重疊。
    x_offset = {
        "observed": -0.08,
        "predicted": 0.08,
    }

    for row_index, model_name in enumerate(
        MODEL_ORDER
    ):
        for col_index, outcome in enumerate(
            OUTCOMES
        ):
            ax = axes[
                row_index,
                col_index,
            ]

            sub = plot_df[
                (
                    plot_df["model"]
                    == model_name
                )
                & (
                    plot_df["outcome"]
                    == outcome
                )
            ].copy()

            for group_name in [
                "Control",
                "Statin",
            ]:
                ss = (
                    sub[
                        sub["Group"]
                        == group_name
                    ]
                    .sort_values(
                        "Time_Num"
                    )
                    .copy()
                )

                if ss.empty:
                    continue

                if x_var == "Time_mid":
                    ss = (
                        ss.dropna(
                            subset=[
                                "Time_mid"
                            ]
                        )
                        .sort_values(
                            "Time_mid"
                        )
                    )

                    x_base = (
                        ss["Time_mid"]
                        .to_numpy(
                            dtype=float
                        )
                    )

                    # continuous-time 軸的 offset
                    # 以年為單位，僅作視覺錯開。
                    observed_x = (
                        x_base
                        - 0.015
                    )

                    predicted_x = (
                        x_base
                        + 0.015
                    )

                    xlabel = (
                        xlabel_year
                    )

                else:
                    x_base = (
                        ss["Time_Num"]
                        .to_numpy(
                            dtype=float
                        )
                    )

                    observed_x = (
                        x_base
                        + x_offset[
                            "observed"
                        ]
                    )

                    predicted_x = (
                        x_base
                        + x_offset[
                            "predicted"
                        ]
                    )

                    xlabel = (
                        xlabel_period
                    )

                ax.scatter(
                    observed_x,
                    ss["true_mean"],
                    marker=observed_marker,
                    s=24,
                    facecolors="none",
                    edgecolors=colors[
                        group_name
                    ],
                    linewidths=1.1,
                    alpha=0.95,
                    label=(
                        f"{group_labels[group_name]}: "
                        f"{observed_label}"
                    ),
                    zorder=3,
                )

                ax.scatter(
                    predicted_x,
                    ss["pred_mean"],
                    marker=predicted_marker,
                    s=24,
                    color=colors[
                        group_name
                    ],
                    alpha=0.90,
                    label=(
                        f"{group_labels[group_name]}: "
                        f"{predicted_label}"
                    ),
                    zorder=4,
                )

            ax.set_title(
                (
                    f"{model_name}: "
                    f"{OUTCOME_LABELS[outcome]}"
                ),
                fontsize=10,
                fontweight="bold",
            )

            ax.set_ylabel(
                OUTCOME_LABELS[
                    outcome
                ]
            )

            ax.set_xlabel(
                xlabel
            )

            ax.grid(
                alpha=0.25
            )

            if x_var == "Time_Num":
                ax.set_xticks(
                    range(
                        1,
                        20,
                    )
                )

                ax.set_xticklabels(
                    [
                        f"T{i}"
                        for i in range(
                            1,
                            20,
                        )
                    ],
                    fontsize=7,
                )

    all_handles = []
    all_labels = []

    for ax in axes.flat:
        handles, labels = (
            ax.get_legend_handles_labels()
        )

        all_handles.extend(
            handles
        )

        all_labels.extend(
            labels
        )

    unique_legend = {}

    for handle, label in zip(
        all_handles,
        all_labels,
    ):
        if (
            label
            and label
            not in unique_legend
        ):
            unique_legend[
                label
            ] = handle

    fig.legend(
        unique_legend.values(),
        unique_legend.keys(),
        loc="lower center",
        bbox_to_anchor=(
            0.5,
            0.055,
        ),
        ncol=4,
        frameon=False,
        fontsize=8,
    )

    title = (
        title_year
        if x_var
        == "Time_mid"
        else title_period
    )

    note = (
        note_year
        if x_var
        == "Time_mid"
        else note_period
    )

    tag = (
        "continuous_time"
        if x_var
        == "Time_mid"
        else "T1_T19"
    )

    fig.suptitle(
        title,
        fontsize=15,
        fontweight="bold",
    )

    fig.text(
        0.5,
        0.015,
        note,
        ha="center",
        va="bottom",
        fontsize=8,
        wrap=True,
    )

    fig.tight_layout(
        rect=[
            0,
            0.11,
            1,
            0.94,
        ]
    )

    out_file = (
        FIGURE_DIR
        / (
            "dl_observed_vs_predicted_"
            f"points_12panel_{tag}_{suffix}.png"
        )
    )

    fig.savefig(
        out_file,
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none",
    )

    plt.close(fig)

    return out_file


figure_files = []

for language in [
    "zh",
    "en",
]:
    figure_files.append(
        plot_loss_curves_12panel(
            dl_loss_history_by_epoch,
            language,
        )
    )

    figure_files.append(
        plot_prediction_points_12panel(
            dl_test_predictions_individual,
            "Time_Num",
            language,
        )
    )

    figure_files.append(
        plot_prediction_points_12panel(
            dl_test_predictions_individual,
            "Time_mid",
            language,
        )
    )

print(
    "已輸出圖檔："
)

for file_path in figure_files:
    print(
        file_path
    )

已輸出圖檔：
analysis_outputs\dl_output\figures\dl_loss_curves_12panel_zh.png
analysis_outputs\dl_output\figures\dl_observed_vs_predicted_points_12panel_T1_T19_zh.png
analysis_outputs\dl_output\figures\dl_observed_vs_predicted_points_12panel_continuous_time_zh.png
analysis_outputs\dl_output\figures\dl_loss_curves_12panel_en.png
analysis_outputs\dl_output\figures\dl_observed_vs_predicted_points_12panel_T1_T19_en.png
analysis_outputs\dl_output\figures\dl_observed_vs_predicted_points_12panel_continuous_time_en.png


In [9]:
# ============================================================
# 8. 最終自動檢查與完成訊息
# ============================================================

required_metric_groups = set(GROUP_LEVELS)
required_models = set(MODEL_ORDER)
required_outcomes = set(OUTCOMES)

# ------------------------------------------------------------
# 單一 prediction-safe dataset 檢查
# ------------------------------------------------------------

# 方案 A 的 summary 表使用 dataset_n，
# 不再使用舊版多重插補流程的 n_imp。
assert "dataset_n" in dl_prediction_metrics_summary_group.columns
assert dl_prediction_metrics_summary_group["dataset_n"].eq(
    EXPECTED_M
).all()

# 內部原始指標表仍保留 .imp 欄位以維持上游資料鍵，
# 但目前只能有一個 prediction-safe dataset。
assert dl_prediction_metrics_raw[".imp"].nunique() == EXPECTED_M

# ------------------------------------------------------------
# 測試真值與預測值檢查
# ------------------------------------------------------------

assert truth_consistency["passed"].all()

assert dl_test_predictions_individual[
    "target_originally_observed"
].all()

assert dl_test_predictions_individual["y_true"].gt(0).all()
assert dl_test_predictions_individual["y_pred"].gt(0).all()
assert mape_denominator_qc["near_zero_n"].eq(0).all()
assert np.isfinite(
    mape_denominator_qc[
        ["min_y_true", "q01_y_true"]
    ].to_numpy(dtype=float)
).all()

assert not ode_target_gap_qc.empty

assert np.isfinite(
    ode_target_gap_qc[
        "target_gap_from_last_history"
    ].to_numpy(dtype=float)
).all()

assert ode_target_gap_qc[
    "target_gap_from_last_history"
].ge(0).all()

assert np.isfinite(
    dl_test_predictions_individual[
        ["y_true", "y_pred", "error", "abs_error", "ape"]
    ].to_numpy(dtype=float)
).all()

# ------------------------------------------------------------
# 模型、結果變項與分組完整性
# ------------------------------------------------------------

assert set(
    dl_prediction_metrics_summary_group[
        "Group_eval"
    ].astype(str)
) == required_metric_groups

assert set(
    dl_prediction_metrics_summary_group[
        "model"
    ].astype(str)
) == required_models

assert set(
    dl_prediction_metrics_summary_group[
        "outcome"
    ].astype(str)
) == required_outcomes

# ------------------------------------------------------------
# 個體預測唯一性
# ------------------------------------------------------------

assert not dl_test_predictions_individual.duplicated(
    [".imp", "model", "outcome", "row_id"]
).any()

# ------------------------------------------------------------
# 誤差欄位重新計算檢查
# ------------------------------------------------------------

error_recalc = (
    dl_test_predictions_individual[
        "y_pred"
    ].to_numpy(dtype=float)
    -
    dl_test_predictions_individual[
        "y_true"
    ].to_numpy(dtype=float)
)

assert np.allclose(
    dl_test_predictions_individual[
        "error"
    ].to_numpy(dtype=float),
    error_recalc,
    rtol=0.0,
    atol=1e-10,
)

assert np.allclose(
    dl_test_predictions_individual[
        "abs_error"
    ].to_numpy(dtype=float),
    np.abs(error_recalc),
    rtol=0.0,
    atol=1e-10,
)

assert np.allclose(
    dl_test_predictions_individual[
        "ape"
    ].to_numpy(dtype=float),
    np.abs(
        error_recalc
        /
        dl_test_predictions_individual[
            "y_true"
        ].to_numpy(dtype=float)
    ) * 100.0,
    rtol=0.0,
    atol=1e-10,
)

# ------------------------------------------------------------
# 各模型樣本數一致性檢查
# ------------------------------------------------------------

sample_size_invariance = (
    dl_sequence_sample_size
    .groupby(
        ["model", "outcome"],
        as_index=False
    )
    .agg(
        train_min=("train_target_n", "min"),
        train_max=("train_target_n", "max"),
        validation_min=("validation_target_n", "min"),
        validation_max=("validation_target_n", "max"),
        test_min=("test_target_n", "min"),
        test_max=("test_target_n", "max"),
        n_dataset=(".imp", "nunique"),
    )
)

assert sample_size_invariance[
    "n_dataset"
].eq(EXPECTED_M).all()

assert sample_size_invariance[
    "train_min"
].eq(
    sample_size_invariance["train_max"]
).all()

assert sample_size_invariance[
    "validation_min"
].eq(
    sample_size_invariance["validation_max"]
).all()

assert sample_size_invariance[
    "test_min"
].eq(
    sample_size_invariance["test_max"]
).all()

print("深度學習分析程式已執行完成，最終自動檢查通過。")
print("目前結果可供修正版資料分析.R 讀取。")
print("主要輸出：")
print("1.", OUT_DIR / "dl_prediction_metrics_summary_group.csv")
print("2.", TABLE_DIR / "dl_test_predictions_individual.csv")
print("3.", TABLE_DIR / "dl_target_manifest.csv")
print("4.", TABLE_DIR / "dl_sequence_sample_size.csv")
print("5.", TABLE_DIR / "DL_model_outputs_type2.xlsx")
print("6.", TABLE_DIR / "dl_trajectory_interval_counts_and_means.csv")
print("7.", TABLE_DIR / "dl_ode_target_gap_summary.csv")
print("8.", TABLE_DIR / "dl_mape_denominator_qc.csv")


深度學習分析程式已執行完成，最終自動檢查通過。
目前結果可供修正版資料分析.R 讀取。
主要輸出：
1. analysis_outputs\dl_output\dl_prediction_metrics_summary_group.csv
2. analysis_outputs\dl_output\tables\dl_test_predictions_individual.csv
3. analysis_outputs\dl_output\tables\dl_target_manifest.csv
4. analysis_outputs\dl_output\tables\dl_sequence_sample_size.csv
5. analysis_outputs\dl_output\tables\DL_model_outputs_type2.xlsx
6. analysis_outputs\dl_output\tables\dl_trajectory_interval_counts_and_means.csv
7. analysis_outputs\dl_output\tables\dl_ode_target_gap_summary.csv
8. analysis_outputs\dl_output\tables\dl_mape_denominator_qc.csv
